# GraphRAG de Microsoft: Indexación y Búsqueda

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/graphrag/02-graphrag-microsoft.ipynb)

Pipeline completo de GraphRAG: preparación de documentos, indexación y búsqueda global/local.

## ¿Qué aprenderás?

- Diferencias entre búsqueda **global** (temas transversales) y **local** (entidades específicas)
- Configurar GraphRAG con Claude como LLM
- Ejecutar el pipeline de indexación
- Usar la API programática para consultas
- Estimar costes antes de lanzar la indexación

## Arquitectura de GraphRAG

```
Documentos
    │
    ▼
Extracción de entidades y relaciones (LLM)
    │
    ▼
Grafo de conocimiento + Comunidades (Leiden)
    │
    ├── Búsqueda LOCAL  → entidades, vecinos inmediatos
    └── Búsqueda GLOBAL → resúmenes de comunidades
```

## Aviso de costes

La indexación hace **muchas llamadas LLM**. Para 100 documentos medianos, espera ~$1-5 USD con GPT-4 Mini o claude-haiku. Siempre estima antes de indexar.

In [ ]:
# Instalar GraphRAG y dependencias
# pip install graphrag anthropic tiktoken pandas

import os
import json
import subprocess
import shutil
import tempfile
from pathlib import Path
from datetime import datetime

import pandas as pd

# Directorio de trabajo para el proyecto GraphRAG
GRAPHRAG_ROOT = Path(tempfile.mkdtemp(prefix="graphrag_demo_"))
INPUT_DIR = GRAPHRAG_ROOT / "input"
OUTPUT_DIR = GRAPHRAG_ROOT / "output"
PROMPTS_DIR = GRAPHRAG_ROOT / "prompts"

for d in [INPUT_DIR, OUTPUT_DIR, PROMPTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Directorio de trabajo: {GRAPHRAG_ROOT}")
print(f"Estructura creada:")
for p in GRAPHRAG_ROOT.rglob("*"):
    print(f"  {p.relative_to(GRAPHRAG_ROOT)}")

## 1. Crear `settings.yaml` con Claude como LLM

GraphRAG soporta cualquier LLM compatible con OpenAI. Aquí configuramos Claude a través de la API de Anthropic.
Usamos `claude-haiku-4-5-20251001` para minimizar costes durante la indexación.

In [ ]:
SETTINGS_YAML = """
# GraphRAG settings.yaml — configurado para Claude via Anthropic API

encoding_model: cl100k_base
skip_workflows: []

llm:
  api_key: ${ANTHROPIC_API_KEY}
  type: openai_chat           # GraphRAG usa interfaz OpenAI-compatible
  model: claude-haiku-4-5-20251001
  api_base: https://api.anthropic.com/v1
  max_tokens: 4096
  temperature: 0.0
  request_timeout: 180.0
  max_retries: 10
  max_retry_wait: 30.0

parallelization:
  stagger: 0.3
  num_threads: 4

async_mode: threaded

embeddings:
  async_mode: threaded
  llm:
    api_key: ${ANTHROPIC_API_KEY}
    type: openai_embedding
    model: text-embedding-3-small   # OpenAI embeddings como alternativa
    api_base: https://api.openai.com/v1

chunks:
  size: 1200
  overlap: 100
  group_by_columns: [id]

input:
  type: file
  file_type: text
  base_dir: input

cache:
  type: file
  base_dir: cache

storage:
  type: file
  base_dir: output

reporting:
  type: file
  base_dir: reports

entity_extraction:
  max_gleanings: 1

community_reports:
  max_length: 2000
  max_input_length: 8000

cluster_graph:
  max_cluster_size: 10

umap:
  enabled: false

snapshots:
  graphml: true
  raw_entities: true
  top_level_nodes: true
"""

settings_path = GRAPHRAG_ROOT / "settings.yaml"
with open(settings_path, "w", encoding="utf-8") as f:
    f.write(SETTINGS_YAML)

print(f"settings.yaml creado en: {settings_path}")
print(f"Tamaño: {settings_path.stat().st_size} bytes")

## 2. Preparar documentos de prueba

Creamos un corpus de textos sobre inteligencia artificial para demostrar la indexación.
En un caso real, estos serían tus documentos corporativos, papers, etc.

In [ ]:
DOCUMENTOS = [
    {
        "nombre": "historia_ia.txt",
        "contenido": """Historia de la Inteligencia Artificial

La inteligencia artificial (IA) tiene sus raíces en la conferencia de Dartmouth de 1956, donde
John McCarthy, Marvin Minsky, Claude Shannon y otros pioneros sentaron las bases del campo.
McCarthy acuñó el término "inteligencia artificial" y desarrolló el lenguaje LISP.

En los años 60 y 70, los sistemas expertos como MYCIN (medicina) y DENDRAL (química) demostraron
que las máquinas podían resolver problemas especializados. Sin embargo, las limitaciones
computacionales llevaron al llamado "invierno de la IA" en los años 80.

El renacimiento llegó con el aprendizaje profundo. En 2012, AlexNet de Krizhevsky, Sutskever
e Hinton ganó el concurso ImageNet con una ventaja sin precedentes, marcando el inicio de la
era moderna de la IA basada en redes neuronales profundas.

Los transformers, introducidos por Vaswani et al. en 2017 con el paper "Attention is All You Need",
revolucionaron el procesamiento del lenguaje natural. Esta arquitectura es la base de GPT,
Claude, Gemini y todos los grandes modelos de lenguaje actuales.
"""
    },
    {
        "nombre": "graphrag_tecnica.txt",
        "contenido": """GraphRAG: Recuperación Aumentada por Grafos

GraphRAG es una técnica desarrollada por Microsoft Research que combina grafos de conocimiento
con modelos de lenguaje grande (LLMs) para mejorar la recuperación de información.

El sistema fue presentado por Darren Edge, Ha Trinh, Newman Cheng y Jonathan Larson en 2024.
A diferencia del RAG vectorial tradicional, GraphRAG construye una jerarquía de comunidades
usando el algoritmo de Leiden para clustering de grafos.

El pipeline de indexación tiene tres fases principales:
1. Extracción: el LLM identifica entidades (personas, organizaciones, conceptos) y relaciones
2. Clustering: las entidades se agrupan en comunidades mediante el algoritmo de Leiden
3. Resúmenes: el LLM genera resúmenes de cada comunidad para búsqueda global

La búsqueda global responde preguntas sobre temas amplios consultando los resúmenes de comunidades.
La búsqueda local responde preguntas sobre entidades específicas explorando el subgrafo local.

Microsoft publicó GraphRAG como software de código abierto en GitHub bajo la organización
microsoft/graphrag, con licencia MIT.
"""
    },
    {
        "nombre": "empresas_ia.txt",
        "contenido": """Empresas Líderes en Inteligencia Artificial

OpenAI, fundada en 2015 por Sam Altman, Elon Musk, Greg Brockman y otros, desarrolla los
modelos GPT y el asistente ChatGPT. En 2023 lanzó GPT-4, considerado el modelo más capaz
en tareas de razonamiento. OpenAI tiene una alianza estratégica con Microsoft, que ha
invertido más de 13.000 millones de dólares en la compañía.

Anthropic fue fundada en 2021 por Dario Amodei y Daniela Amodei, ex empleados de OpenAI.
La empresa desarrolla Claude, un asistente de IA con enfoque en seguridad y alineamiento.
Claude usa técnicas de RLHF (Reinforcement Learning from Human Feedback) y aprendizaje
constitucional para comportarse de forma segura y útil.

Google DeepMind, resultado de la fusión de Google Brain y DeepMind en 2023, desarrolla
Gemini y mantiene investigación en AlphaFold (proteínas), AlphaCode (código) y Gemini.
Demis Hassabis lidera la división tras la fusión.

Meta AI, liderada por Yann LeCun, ha apostado por modelos de código abierto con la familia
LLaMA. LLaMA 3 puede descargarse libremente y ha democratizado el acceso a modelos potentes.
"""
    },
]

for doc in DOCUMENTOS:
    ruta = INPUT_DIR / doc["nombre"]
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(doc["contenido"])
    print(f"Creado: {ruta.name} ({len(doc['contenido'])} caracteres)")

print(f"\nTotal: {len(DOCUMENTOS)} documentos en {INPUT_DIR}")

## 3. Ejecutar indexación

La indexación real requiere `graphrag` instalado y credenciales de API válidas.
Aquí mostramos el comando y simulamos el resultado esperado.

In [ ]:
def ejecutar_indexacion(root_dir: Path, dry_run: bool = True) -> dict:
    """
    Ejecuta el pipeline de indexación de GraphRAG.
    
    Args:
        root_dir: directorio raíz del proyecto GraphRAG
        dry_run: si True, solo muestra el comando sin ejecutarlo
    
    Returns:
        dict con resultado de la indexación
    """
    comando = [
        "python", "-m", "graphrag", "index",
        "--root", str(root_dir),
        "--verbose",
    ]

    print("Comando de indexación:")
    print(" ".join(comando))
    print()

    if dry_run:
        print("[DRY RUN] Indexación simulada. Establece ANTHROPIC_API_KEY y dry_run=False para ejecutar.")
        return {
            "status": "simulado",
            "entidades_extraidas": 18,
            "relaciones_extraidas": 24,
            "comunidades": 4,
            "chunks_procesados": 9,
            "tiempo_segundos": 0,
        }

    # Ejecución real
    inicio = datetime.now()
    env = os.environ.copy()
    resultado = subprocess.run(
        comando,
        capture_output=True,
        text=True,
        cwd=str(root_dir),
        env=env,
    )
    duracion = (datetime.now() - inicio).total_seconds()

    if resultado.returncode != 0:
        print("ERROR en indexación:")
        print(resultado.stderr)
        return {"status": "error", "stderr": resultado.stderr}

    print("Indexación completada.")
    print(resultado.stdout[-2000:])  # últimas líneas del log
    return {"status": "ok", "tiempo_segundos": duracion}


resultado_indexacion = ejecutar_indexacion(GRAPHRAG_ROOT, dry_run=True)
print("\nResultado:")
for k, v in resultado_indexacion.items():
    print(f"  {k}: {v}")

## 4. Cliente GraphRAG para búsqueda programática

Implementamos `GraphRAGClient` que encapsula las búsquedas global y local con modo simulado.

In [ ]:
class GraphRAGClient:
    """
    Cliente para consultas GraphRAG (global y local).
    
    En modo real, invoca los comandos CLI de graphrag.
    En modo simulado, devuelve respuestas ilustrativas.
    """

    RESPUESTAS_SIMULADAS = {
        "global": {
            "default": (
                "[BÚSQUEDA GLOBAL SIMULADA]\n\n"
                "Basándome en los resúmenes de comunidades del grafo, los temas principales son:\n"
                "1. Pioneros de la IA (McCarthy, Minsky, Shannon) y la conferencia de Dartmouth (1956)\n"
                "2. La revolución del deep learning (AlexNet 2012, Transformers 2017)\n"
                "3. Empresas líderes: OpenAI, Anthropic, Google DeepMind, Meta AI\n"
                "4. GraphRAG como técnica de recuperación de información estructurada\n"
            )
        },
        "local": {
            "default": (
                "[BÚSQUEDA LOCAL SIMULADA]\n\n"
                "Entidades relacionadas encontradas en el subgrafo local:\n"
                "- Anthropic: empresa fundada por Dario y Daniela Amodei en 2021\n"
                "- Claude: asistente IA con enfoque en seguridad y alineamiento\n"
                "- Relación: Anthropic -> DESARROLLA -> Claude\n"
                "- Contexto: ex empleados de OpenAI, técnicas RLHF y Constitutional AI\n"
            )
        },
    }

    def __init__(self, root_dir: Path, modo_simulado: bool = True):
        self.root_dir = root_dir
        self.modo_simulado = modo_simulado

    def busqueda_global(self, pregunta: str) -> str:
        """Búsqueda global: responde preguntas sobre temas transversales."""
        if self.modo_simulado:
            return self.RESPUESTAS_SIMULADAS["global"]["default"]

        resultado = subprocess.run(
            ["python", "-m", "graphrag", "query",
             "--root", str(self.root_dir),
             "--method", "global",
             "--query", pregunta],
            capture_output=True, text=True, cwd=str(self.root_dir)
        )
        return resultado.stdout if resultado.returncode == 0 else resultado.stderr

    def busqueda_local(self, pregunta: str) -> str:
        """Búsqueda local: responde sobre entidades y sus relaciones directas."""
        if self.modo_simulado:
            return self.RESPUESTAS_SIMULADAS["local"]["default"]

        resultado = subprocess.run(
            ["python", "-m", "graphrag", "query",
             "--root", str(self.root_dir),
             "--method", "local",
             "--query", pregunta],
            capture_output=True, text=True, cwd=str(self.root_dir)
        )
        return resultado.stdout if resultado.returncode == 0 else resultado.stderr


cliente = GraphRAGClient(GRAPHRAG_ROOT, modo_simulado=True)
print("GraphRAGClient listo en modo simulado.")

## 5. Demo: búsqueda global vs local

La diferencia clave:
- **Global**: síntesis de toda la base de conocimiento, ideal para preguntas amplias
- **Local**: exploración del entorno inmediato de una entidad, ideal para preguntas específicas

In [ ]:
preguntas_globales = [
    "¿Cuáles son los principales hitos en la historia de la IA?",
    "¿Qué empresas lideran el mercado de la inteligencia artificial?",
]

preguntas_locales = [
    "¿Qué es Anthropic y quiénes la fundaron?",
    "¿Qué relación tiene GraphRAG con Microsoft?",
]

print("=" * 70)
print("BÚSQUEDA GLOBAL (síntesis del corpus completo)")
print("=" * 70)
for pregunta in preguntas_globales:
    print(f"\nPregunta: {pregunta}")
    print("-" * 50)
    respuesta = cliente.busqueda_global(pregunta)
    print(respuesta)

print()
print("=" * 70)
print("BÚSQUEDA LOCAL (entidades y relaciones directas)")
print("=" * 70)
for pregunta in preguntas_locales:
    print(f"\nPregunta: {pregunta}")
    print("-" * 50)
    respuesta = cliente.busqueda_local(pregunta)
    print(respuesta)

## 6. Análisis de coste estimado

Antes de indexar un corpus real, conviene estimar cuántas llamadas LLM se harán y qué costarán.

In [ ]:
def estimar_coste_indexacion(
    n_documentos: int,
    palabras_promedio: int = 500,
    modelo: str = "claude-haiku-4-5-20251001",
) -> dict:
    """
    Estima el coste de indexación en USD.
    
    Fórmula simplificada:
    - Extracción de entidades: ~2 llamadas LLM por chunk
    - Chunks = palabras_totales / 300 (aprox 300 palabras por chunk)
    - Resúmenes de comunidades: ~10% de los chunks
    """
    # Precios en USD por 1M tokens (entrada / salida)
    precios = {
        "claude-haiku-4-5-20251001": {"entrada": 0.25, "salida": 1.25},
        "claude-sonnet-4-5": {"entrada": 3.00, "salida": 15.00},
        "gpt-4o-mini": {"entrada": 0.15, "salida": 0.60},
    }

    precio = precios.get(modelo, precios["claude-haiku-4-5-20251001"])

    total_palabras = n_documentos * palabras_promedio
    tokens_totales = total_palabras * 1.3  # ~1.3 tokens por palabra en español
    n_chunks = int(total_palabras / 300) + 1
    llamadas_extraccion = n_chunks * 2
    llamadas_resumenes = max(1, n_chunks // 10)

    tokens_entrada_extraccion = llamadas_extraccion * 1500  # ~1500 tokens de contexto por llamada
    tokens_salida_extraccion = llamadas_extraccion * 500
    tokens_entrada_resumen = llamadas_resumenes * 3000
    tokens_salida_resumen = llamadas_resumenes * 800

    coste_extraccion = (
        tokens_entrada_extraccion / 1_000_000 * precio["entrada"]
        + tokens_salida_extraccion / 1_000_000 * precio["salida"]
    )
    coste_resumen = (
        tokens_entrada_resumen / 1_000_000 * precio["entrada"]
        + tokens_salida_resumen / 1_000_000 * precio["salida"]
    )
    coste_total = coste_extraccion + coste_resumen

    return {
        "modelo": modelo,
        "documentos": n_documentos,
        "chunks_estimados": n_chunks,
        "llamadas_llm_estimadas": llamadas_extraccion + llamadas_resumenes,
        "coste_extraccion_usd": round(coste_extraccion, 4),
        "coste_resumenes_usd": round(coste_resumen, 4),
        "coste_total_usd": round(coste_total, 4),
        "coste_total_eur": round(coste_total * 0.93, 4),
    }


# Comparativa de escenarios
escenarios = [
    (3, 300, "claude-haiku-4-5-20251001"),
    (50, 500, "claude-haiku-4-5-20251001"),
    (200, 800, "claude-haiku-4-5-20251001"),
    (200, 800, "claude-sonnet-4-5"),
    (200, 800, "gpt-4o-mini"),
]

filas = [estimar_coste_indexacion(n, p, m) for n, p, m in escenarios]
df_costes = pd.DataFrame(filas)

print("=== Estimación de costes de indexación ===")
columnas = ["modelo", "documentos", "chunks_estimados", "llamadas_llm_estimadas",
            "coste_total_usd", "coste_total_eur"]
print(df_costes[columnas].to_string(index=False))

print("\nConsejo: usa claude-haiku para indexacion y claude-sonnet para consultas en produccion.")

## Resumen

### Pipeline completo de GraphRAG

| Paso | Herramienta | Descripción |
|------|-------------|-------------|
| Preparación | Python | Organizar documentos en `input/` |
| Configuración | `settings.yaml` | LLM, embeddings, chunking |
| Indexación | `graphrag index` | Extrae entidades, construye grafo, genera resúmenes |
| Búsqueda global | `graphrag query --method global` | Preguntas temáticas amplias |
| Búsqueda local | `graphrag query --method local` | Preguntas sobre entidades específicas |

### Cuándo usar cada tipo de búsqueda

| Tipo | Mejor para | Ejemplo |
|------|-----------|------|
| **Global** | Visión de conjunto, tendencias | "¿Qué temas dominan el corpus?" |
| **Local** | Entidades concretas, relaciones | "¿Qué proyectos tiene empresa X?" |

### Próximo paso

En el **Notebook 03** veremos cómo extraer grafos de conocimiento automáticamente con Claude usando `tool_use`.